# Chatbot evaluation

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [8]:
from langsmith import Client

client = Client()

# Define dataset: these are the test cases
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['e8575422-b709-41a2-8239-3593371277c3',
  '3f274006-198c-4e12-b38d-33c81b9c70e4',
  '1df62ba6-168e-4db4-8592-5c0170986d73',
  'f62d997e-7e22-40e3-b51f-6437d8dc83c7',
  '7f2e3834-c21b-4b9e-9358-6621da37aba3'],
 'count': 5,
 'as_of': '2026-03-04T07:11:19.121993703Z'}

In [10]:
## LLM as Judge

import openai
from langsmith import wrappers
 
openai_client=wrappers.wrap_openai(openai.OpenAI())

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict,outputs:dict, reference_outputs:dict)->bool:
      user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
      response=openai_client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                  {"role":"system","content":eval_instructions},
                  {"role":"user","content":user_content}
            ]
      ).choices[0].message.content

      return response == "CORRECT"

In [11]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

In [12]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model: str = "gpt-4o-mini", instructions: str = default_instructions) -> str:
    return openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content


In [13]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [15]:
## Run our evaluation
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-4o-mini-chatbot"
)

View the evaluation results for experiment: 'openai-4o-mini-chatbot-382f35b5' at:
https://smith.langchain.com/o/50c65426-f9cd-43bf-8113-206780180ab4/datasets/3fb7beaa-91a2-4fdf-b6a0-2a46da372f6e/compare?selectedSessions=2ee03ffa-a975-4254-a880-43c5c1a7e4da




5it [00:12,  2.41s/it]


In [17]:
### Call my_app for every datapoints
def ls_target_4o(inputs: str) -> dict:
    return {"response": my_app(inputs["question"],model="gpt-4o")}

In [19]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target_4o,
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-4o-chatbot"
)

View the evaluation results for experiment: 'openai-4o-chatbot-d866c680' at:
https://smith.langchain.com/o/50c65426-f9cd-43bf-8113-206780180ab4/datasets/3fb7beaa-91a2-4fdf-b6a0-2a46da372f6e/compare?selectedSessions=bea086e3-334a-4cd6-89a1-ba729a504c48




5it [00:09,  1.90s/it]
